In [16]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import os
import numpy as np
from PIL import Image

# Wybranie i przekopiowanie 100 losowych autentycznych zdjęć do zbioru testowego
# path = "../../../Dane/Humans"
# files = os.listdir(path)
# 
# for i in range(100):
#     random_file = np.random.choice(files)
#     img = Image.open(os.path.join(path, random_file))
#     img.save(os.path.join('../../../Dane/BezelAI/train/original', random_file))
#     
# # Wybranie 100 zdjęc podstawionych
# path2 = "../../../Dane/Testy projektu/zdjecia-uporzadkowane2/oszust-smartfon-2"
# 
# files2 = os.listdir(path2)
# 
# for i in range(100):
#     img2 = np.random.choice(files2)
#     
#     img2 = Image.open(os.path.join(path2, img2))
#     
#     img2.save(os.path.join('../../../Dane/BezelAI/train/spoof', f"spoof_{i}.jpg"))

In [17]:
# Wczytanie danych i zmiana rozmiaru
train_path = '../../../Dane/BezelAI/train'
val_path = '../../../Dane/BezelAI/val'

train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

train_dataset = datasets.ImageFolder(root=train_path, transform=train_transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_dataset = datasets.ImageFolder(root=val_path, transform=val_transform)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("Train: ", train_dataset.class_to_idx)
print("Val: ", val_dataset.class_to_idx)

In [18]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Definicja modelu sieci neuronowej
# class BezelDetectionNet(nn.Module):
#     def __init__(self):
#         super(BezelDetectionNet, self).__init__()
#         # Conv -> ReLU -> Pool -> Conv -> ReLU -> Flatten -> Fully connected -> Fully connected
#         self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
#         self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
#         self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
#         self.fc1 = nn.Linear(64 * 32 * 32, 512)
#         self.fc2 = nn.Linear(512, 2)    # 2 klasy na wyjściu (original, spoof)
# 
#     # Definicja przepływu danych przez sieć
#     def forward(self, x):
#         x = self.pool(F.relu(self.conv1(x)))
#         x = self.pool(F.relu(self.conv2(x)))
#         x = x.view(-1, 64 * 32 * 32)
#         x = F.relu(self.fc1(x))
#         x = self.fc2(x)
#         return x

# Eksperymenty
class BezelDetectionNet(nn.Module):
    def __init__(self):
        super(BezelDetectionNet, self).__init__()
        # Conv -> ReLU -> Pool -> Conv -> ReLU -> Flatten -> Fully connected -> Fully connected
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1)
        self.bn4 = nn.BatchNorm2d(256)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
        self.fc1 = nn.Linear(256 * 8 * 8, 1024)
        self.fc2 = nn.Linear(1024, 512)
        self.fc3 = nn.Linear(512, 2)    # 2 klasy na wyjściu (original, spoof)

    # Definicja przepływu danych przez sieć
    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = self.pool(F.relu(self.bn3(self.conv3(x))))
        x = self.pool(F.relu(self.bn4(self.conv4(x))))
        x = x.view(-1, 256 * 8 * 8)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = BezelDetectionNet()

In [19]:
import torch.optim as optim

# Trenowaie modelu
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Pętla treningowa
for epoch in range(10):
    for i, data in enumerate(train_loader, 0):
        inputs, labels = data
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

In [20]:
# Sprawdzenie skuteczności modelu
correct = 0
total = 0
with torch.no_grad():
    for data in val_loader:
        images, labels = data
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Dokładność na zbiorze walidacyjnym: {100 * correct // total} %')

In [21]:
# Zapisanie modelu
torch.save(model.state_dict(), "bezelai_model.pth")
torch.save(model, "bezelai_model_full.pth")